In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
import numpy as np

def glasplatte_auf_rohr():
    # Geometrie [mm]
    Lx = 1000.0         # Plattenlänge
    Ly = 500.0          # Plattenbreite
    t  = 1.0            # Glasdicke

    cx = Lx / 2.0       # Mittelpunkt Platte
    cy = Ly / 2.0

    rohr_radius = 80.0      # mittlerer Rohrradius [mm]  <-- anpassen
    rohr_wand   = 0.1       # Wanddicke [mm], nicht 0 für numerische Stabilität

    r_in  = rohr_radius - rohr_wand / 2.0
    r_out = rohr_radius + rohr_wand / 2.0

    maxh_global = 25.0
    maxh_ring   = 2.0       # lokal feiner am Ring

    plate = MoveTo(0, 0).Rectangle(Lx, Ly).Face()
    plate.edges.name = "free"

    outer = Circle((cx, cy), r_out).Face()
    inner = Circle((cx, cy), r_in).Face()
    ring  = outer - inner

    # die beiden Kreisränder des Rings
    ring.edges.name = "support"

    # Ring aus der Platte ausschneiden -> innere Ränder entstehen
    shape = plate - ring

    geo = OCCGeometry(shape, dim=2)
    ngmesh = geo.GenerateMesh(maxh=maxh_global)
    mesh = Mesh(ngmesh)
    mesh.Curve(3)

    E   = 70e6       # N'/mm²  (70 GPa)
    nu  = 0.23
    rho = 2.5e-6     # kg/mm³
    g   = 9810.0     # mm/s²

    q = rho * t * g  # Eigengewicht pro Fläche

    Db = E * t**3 / (12.0 * (1.0 - nu**2))

    def Dinv(A):
        return (1.0 / Db) * (
            (1.0 / (1.0 - nu)) * A
            - (nu / (1.0 - nu**2)) * Trace(A) * Id(2)
        )

    # ------------------------------------------------------------
    # HHJ-Räume
    #
    # w=0 nur auf dem Ring ("support")
    # übrige Ränder frei
    # ------------------------------------------------------------
    order = 3

    V = HDivDiv(mesh, order=order-1, dirichlet="")
    Q = H1(mesh, order=order, dirichlet="support")
    X = V * Q

    (sigma, w), (tau, v) = X.TnT()

    n = specialcf.normal(2)

    def tang(u):
        return u - (u*n)*n

    # ------------------------------------------------------------
    # Bilinearform HHJ
    # ------------------------------------------------------------
    a = BilinearForm(X, symmetric=True)
    a += InnerProduct(Dinv(sigma), tau) * dx
    a += div(sigma) * Grad(v) * dx
    a += div(tau) * Grad(w) * dx
    a += -(sigma[n, :] * tang(Grad(v)) + tau[n, :] * tang(Grad(w))) * dx(element_boundary=True)
    a.Assemble()

    # rechte Seite: Eigengewicht
    f = LinearForm(X)
    f += q * v * dx
    f.Assemble()

    # ------------------------------------------------------------
    # Lösen
    # ------------------------------------------------------------
    gfu = GridFunction(X)
    gfu.vec.data = a.mat.Inverse(X.FreeDofs(), inverse="") * f.vec

    gf_sigma, gf_w = gfu.components

    Draw(gf_w, mesh, name="w")
    Draw(100 * gf_w, mesh, name="disp", deformation=True, euler_angles=[-60, 5, 30])

    # ------------------------------------------------------------
    # Auswertung
    # ------------------------------------------------------------
    step_x = 81
    step_y = 41

    xs = np.linspace(0, Lx, step_x)
    ys = np.linspace(0, Ly, step_y)

    Z = np.zeros((step_y, step_x))
    for iy, yy in enumerate(ys):
        for ix, xx in enumerate(xs):
            try:
                Z[iy, ix] = gf_w(xx, yy)
            except:
                Z[iy, ix] = np.nan

    valid = Z[np.isfinite(Z)]
    print("min w [mm] =", np.min(valid))
    print("max w [mm] =", np.max(valid))
    print("max-min [mm] =", np.max(valid) - np.min(valid))

    # optional XYZ-Export
    filename = "glasplatte_rohrauflager.xyz"
    with open(filename, "w") as out:
        for iy, yy in enumerate(ys):
            for ix, xx in enumerate(xs):
                zz = Z[iy, ix]
                if np.isfinite(zz):
                    out.write(f"{xx:.3f}\t{yy:.3f}\t{zz:.6f}\n")

    return mesh, gf_w, gf_sigma, filename


if __name__ == "__main__":
    mesh, gf_w, gf_sigma, filename = glasplatte_auf_rohr()
    print("XYZ gespeichert:", filename)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': […

min w [mm] = -7.138388401458921
max w [mm] = -1.3659267862700026e-06
max-min [mm] = 7.138387035532135
XYZ gespeichert: glasplatte_rohrauflager.xyz
